# Firth Logistic Comparison: `firthmodels` vs `sse_detection.lib.regression`

This notebook compares the custom Firth logistic implementation in `sse_detection/lib/regression.py` against the installed `firthmodels.FirthLogisticRegression` estimator.

The key design choice is to build one Patsy design matrix per formula and pass that exact matrix to both implementations. `firthmodels` is run with `fit_intercept=False` because the Patsy matrix already includes the intercept column. That keeps coefficient names and ordering aligned.

## Setup

In [3]:
from __future__ import annotations

import sys
from pathlib import Path
from types import SimpleNamespace
from typing import Literal

import numpy as np
import pandas as pd
import patsy

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "utils").exists():
    raise RuntimeError("Run this notebook from inside the scotland repository.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/Users/ydnkka/Desktop/PhD Project/projects/scotland')

In [2]:
import firthmodels
from firthmodels import FirthLogisticRegression

from sse_detection import lib as sselib
from sse_detection.lib import association_pipeline

print("firthmodels module:", firthmodels.__file__)
print("firthmodels version:", getattr(firthmodels, "__version__", "not exposed"))
print("custom fitter:", sselib.fit_custom_firth_logit)

firthmodels module: /opt/homebrew/Caskroom/miniconda/base/envs/PhD/lib/python3.13/site-packages/firthmodels/__init__.py
firthmodels version: not exposed
custom fitter: <function fit_custom_firth_logit at 0x15f5e0400>


## Comparison Helpers

`firthmodels` exposes an sklearn-style estimator; the custom implementation exposes a minimal statsmodels-like result. The helper below converts both into aligned `Series` objects indexed by the Patsy design columns.

In [4]:
def _as_binary_int(y: pd.Series) -> np.ndarray:
    values = y.to_numpy()
    unique = pd.Index(pd.Series(values).dropna().unique())
    if not set(unique).issubset({0, 1, False, True}):
        raise ValueError(f"Expected a binary 0/1 outcome; got values {list(unique)!r}")
    return values.astype(int)


def design_from_formula(data: pd.DataFrame, formula: str):
    y_df, x_df = patsy.dmatrices( # type: ignore
        formula,
        data=data,
        return_type="dataframe",
        NA_action="raise",
    )
    y = _as_binary_int(y_df.iloc[:, 0])
    return y, x_df


def fit_firthmodels_formula(
    data: pd.DataFrame,
    formula: str,
    *,
    backend: Literal['auto', 'numba', 'numpy'] = "numpy",
    max_iter: int = 100,
    xtol: float = 1e-6,
    gtol: float = 1e-6,
):
    y, x_df = design_from_formula(data, formula)
    model = FirthLogisticRegression(
        fit_intercept=False,
        backend=backend,
        max_iter=max_iter,
        xtol=xtol,
        gtol=gtol,
    )
    model.fit(x_df, y)
    return SimpleNamespace(
        estimator=model,
        params=pd.Series(model.coef_, index=x_df.columns, name="firthmodels_estimate"),
        bse=pd.Series(model.bse_, index=x_df.columns, name="firthmodels_std_error"),
        pvalues=pd.Series(model.pvalues_, index=x_df.columns, name="firthmodels_p_value"),
        nobs=len(y),
        n_iter=getattr(model, "n_iter_", np.nan),
        converged=getattr(model, "converged_", np.nan),
        loglik=getattr(model, "loglik_", np.nan),
        design_columns=list(x_df.columns),
        design_rank=int(np.linalg.matrix_rank(x_df.to_numpy(dtype=float))),
    )


def fit_custom_formula(
    data: pd.DataFrame,
    formula: str,
    *,
    maxiter: int = 100,
    tol: float = 1e-6,
):
    result = sselib.fit_custom_firth_logit(data, formula, maxiter=maxiter, tol=tol)
    _, x_df = design_from_formula(data, formula)
    return SimpleNamespace(
        result=result,
        params=result.params.rename("custom_estimate"),
        bse=result.bse.rename("custom_std_error"),
        pvalues=result.pvalues.rename("custom_p_value"),
        nobs=result.nobs,
        n_iter=result.fit_history.get("iterations", np.nan),
        converged=result.converged,
        loglik=result.fit_history.get("penalized_loglike", np.nan),
        design_columns=list(x_df.columns),
        design_rank=int(np.linalg.matrix_rank(x_df.to_numpy(dtype=float))),
    )


def compare_formula(
    data: pd.DataFrame,
    formula: str,
    *,
    label: str,
    backend: Literal['auto', 'numba', 'numpy'] = "numpy",
    max_iter: int = 100,
    tol: float = 1e-6,
)-> tuple[pd.DataFrame, pd.DataFrame]:
    custom = fit_custom_formula(data, formula, maxiter=max_iter, tol=tol)
    reference = fit_firthmodels_formula(
        data,
        formula,
        backend=backend,
        max_iter=max_iter,
        xtol=tol,
        gtol=tol,
    )

    terms = custom.params.index.union(reference.params.index)
    comparison = pd.concat(
        [
            custom.params.reindex(terms),
            reference.params.reindex(terms),
            custom.bse.reindex(terms),
            reference.bse.reindex(terms),
            custom.pvalues.reindex(terms),
            reference.pvalues.reindex(terms),
        ],
        axis=1,
    )
    comparison["estimate_delta"] = (
        comparison["custom_estimate"] - comparison["firthmodels_estimate"]
    )
    comparison["std_error_delta"] = (
        comparison["custom_std_error"] - comparison["firthmodels_std_error"]
    )
    comparison["p_value_delta"] = (
        comparison["custom_p_value"] - comparison["firthmodels_p_value"]
    )
    comparison["abs_estimate_delta"] = comparison["estimate_delta"].abs()
    comparison["abs_std_error_delta"] = comparison["std_error_delta"].abs()

    summary = {
        "label": label,
        "formula": formula,
        "nobs": custom.nobs,
        "n_parameters": len(terms),
        "design_rank": custom.design_rank,
        "full_rank": custom.design_rank == len(terms),
        "custom_converged": bool(custom.converged),
        "firthmodels_converged": bool(reference.converged),
        "custom_iterations": custom.n_iter,
        "firthmodels_iterations": reference.n_iter,
        "max_abs_estimate_delta": float(comparison["abs_estimate_delta"].max()),
        "max_abs_std_error_delta": float(comparison["abs_std_error_delta"].max()),
        "max_abs_p_value_delta": float(comparison["p_value_delta"].abs().max()),
    }
    return comparison, pd.DataFrame([summary])


def show_comparison(
    data: pd.DataFrame,
    formula: str,
    *,
    label: str,
    max_rows: int = 20,
):
    comparison, summary = compare_formula(data, formula, label=label)
    display(summary)
    display(
        comparison.sort_values("abs_estimate_delta", ascending=False)
        .head(max_rows)
        .style.format(precision=6)
    )
    return comparison, summary

## Synthetic Checks

These cases are deliberately small. They are useful for detecting obvious coefficient-ordering, intercept-handling, or convergence mismatches before moving to the SSE association frames.

In [5]:
synthetic_results = []

separated_df = pd.DataFrame(
    {
        "y": [0, 0, 0, 1, 1, 1],
        "x": [0, 0, 0, 1, 1, 1],
    }
)
comparison, summary = show_comparison(
    separated_df,
    "y ~ x",
    label="complete_separation",
)
synthetic_results.append(summary)

,label,formula,nobs,n_parameters,design_rank,full_rank,custom_converged,firthmodels_converged,custom_iterations,firthmodels_iterations,max_abs_estimate_delta,max_abs_std_error_delta,max_abs_p_value_delta
0,complete_separation,y ~ x,6,2,2,True,True,True,13,6,2.155916e-07,0.330764,0.066936


,custom_estimate,firthmodels_estimate,custom_std_error,firthmodels_std_error,custom_p_value,firthmodels_p_value,estimate_delta,std_error_delta,p_value_delta,abs_estimate_delta,abs_std_error_delta
x,3.891821,3.891820,2.468854,2.138090,0.114941,0.068724,0.000000,0.330764,0.046218,0.000000,0.330764
Intercept,-1.945910,-1.945910,1.745743,1.511858,0.264996,0.198060,-0.000000,0.233885,0.066936,0.000000,0.233885


In [6]:
quasi_df = pd.DataFrame(
    {
        "y": [0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1],
        "x": [0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3],
        "group": ["A", "A", "B", "B", "B", "C", "C", "C", "D", "D", "D", "D"],
    }
)
comparison, summary = show_comparison(
    quasi_df,
    "y ~ x + C(group, Treatment(reference='A'))",
    label="quasi_separation_with_categorical_term",
)
synthetic_results.append(summary)

,label,formula,nobs,n_parameters,design_rank,full_rank,custom_converged,firthmodels_converged,custom_iterations,firthmodels_iterations,max_abs_estimate_delta,max_abs_std_error_delta,max_abs_p_value_delta
0,quasi_separation_with_categorical_term,"y ~ x + C(group, Treatment(reference='A'))",12,5,5,True,True,True,22,9,0.000001,0.734646,0.097443


,custom_estimate,firthmodels_estimate,custom_std_error,firthmodels_std_error,custom_p_value,firthmodels_p_value,estimate_delta,std_error_delta,p_value_delta,abs_estimate_delta,abs_std_error_delta
"C(group, Treatment(reference='A'))[T.D]",-0.945571,-0.945570,4.381911,3.647265,0.829152,0.795439,-0.000001,0.734646,0.033713,0.000001,0.734646
"C(group, Treatment(reference='A'))[T.C]",0.104224,0.104225,3.148100,2.608138,0.973589,0.968124,-0.000001,0.539962,0.005465,0.000001,0.539962
"C(group, Treatment(reference='A'))[T.B]",0.066063,0.066063,2.656414,2.181139,0.980159,0.975837,-0.000001,0.475275,0.004322,0.000001,0.475275
x,1.524294,1.524294,1.781210,1.478808,0.392129,0.302654,0.000001,0.302402,0.089475,0.000001,0.302402
Intercept,-1.609438,-1.609438,1.897366,1.549193,0.396300,0.298857,0.000000,0.348173,0.097443,0.000000,0.348173


In [7]:
rng = np.random.default_rng(20260610)
n = 80
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
eta = -0.4 + 0.9 * x1 - 0.5 * x2
p = 1 / (1 + np.exp(-eta))
regular_df = pd.DataFrame(
    {
        "y": rng.binomial(1, p),
        "x1": x1,
        "x2": x2,
    }
)
comparison, summary = show_comparison(
    regular_df,
    "y ~ x1 + x2",
    label="regular_small_sample",
)
synthetic_results.append(summary)

pd.concat(synthetic_results, ignore_index=True)

,label,formula,nobs,n_parameters,design_rank,full_rank,custom_converged,firthmodels_converged,custom_iterations,firthmodels_iterations,max_abs_estimate_delta,max_abs_std_error_delta,max_abs_p_value_delta
0,regular_small_sample,y ~ x1 + x2,80,3,3,True,True,True,6,8,1.177956e-08,0.007419,0.002998


,custom_estimate,firthmodels_estimate,custom_std_error,firthmodels_std_error,custom_p_value,firthmodels_p_value,estimate_delta,std_error_delta,p_value_delta,abs_estimate_delta,abs_std_error_delta
x1,1.166293,1.166293,0.325153,0.318155,0.000335,0.000247,0.000000,0.006998,0.000088,0.000000,0.006998
x2,-0.676994,-0.676994,0.290351,0.282933,0.019720,0.016722,-0.000000,0.007419,0.002998,0.000000,0.007419
Intercept,-0.657690,-0.657690,0.278282,0.273554,0.018108,0.016206,-0.000000,0.004728,0.001903,0.000000,0.004728


,label,formula,nobs,n_parameters,design_rank,full_rank,custom_converged,firthmodels_converged,custom_iterations,firthmodels_iterations,max_abs_estimate_delta,max_abs_std_error_delta,max_abs_p_value_delta
0,complete_separation,y ~ x,6,2,2,True,True,True,13,6,2.155916e-07,0.330764,0.066936
1,quasi_separation_with_categorical_term,"y ~ x + C(group, Treatment(reference='A'))",12,5,5,True,True,True,22,9,1.321436e-06,0.734646,0.097443
2,regular_small_sample,y ~ x1 + x2,80,3,3,True,True,True,6,8,1.177956e-08,0.007419,0.002998


## SSE Association Frame Checks

The cells below compare the two implementations on sampled model frames built from the same association pipeline inputs. Sampling keeps validation interactive while still exercising the real formula structure: candidate outcome, window fixed effects, clade adjustment, and both continuous and categorical predictors.

In [8]:
OUTPUT_DIR = PROJECT_ROOT / "sse_detection" / "results" / "sse_outputs"
MODEL_SETS = sselib.default_model_sets(
    variant_adjuster="clade",
    window_adjustment="fixed_effects",
)
PRIMARY_ADJUSTERS = MODEL_SETS["primary"]

frames = sselib.load_association_frames(
    output_dir=OUTPUT_DIR,
    variant_adjuster="clade",
    group_by_clade=False,
    window_stride=2,
    run_composition=True,
)
node_df = frames.node_model_base.copy()
composition_df = frames.composition_base.copy()

pd.DataFrame(
    [
        {
            "frame": "node_model_base",
            "rows": len(node_df),
            "candidate": int(node_df["candidate"].sum()),
            "background": int((node_df["candidate"] == 0).sum()),
        },
        {
            "frame": "composition_base",
            "rows": len(composition_df),
            "candidate": int(composition_df["candidate"].sum()),
            "background": int((composition_df["candidate"] == 0).sum()),
        },
    ]
)

,frame,rows,candidate,background
0,node_model_base,13059,614,12445
1,composition_base,264139,66840,197299


In [9]:
def formula_variables(terms: list[str]) -> list[str]:
    return sselib.model_variables_from_terms(terms)


def pipeline_like_model_frame(
    source: pd.DataFrame,
    *,
    predictors: list[str],
    adjusters: list[str],
    sample_per_class: int | None = 1200,
    categorical_predictors: bool = False,
    random_state: int = 20260610,
) -> pd.DataFrame:
    required = ["candidate", *predictors, *formula_variables(adjusters)]
    required = list(dict.fromkeys(required))
    d = association_pipeline.complete_case(source, required)
    if categorical_predictors:
        for predictor in predictors:
            d[predictor] = d[predictor].astype(str)

    drop_window = any(term.startswith("C(window_idx") for term in adjusters)
    if drop_window:
        d, _, _ = association_pipeline.drop_nonvarying_levels(d, ["window_idx"])

    if sample_per_class is not None:
        sampled = []
        for _, group in d.groupby("candidate", dropna=False):
            n = min(sample_per_class, len(group))
            sampled.append(group.sample(n=n, random_state=random_state))
        d = pd.concat(sampled, ignore_index=False).sample(
            frac=1,
            random_state=random_state,
        )
        if drop_window:
            d, _, _ = association_pipeline.drop_nonvarying_levels(d, ["window_idx"])

    if d["candidate"].nunique(dropna=True) != 2:
        raise ValueError("Prepared comparison frame does not contain both outcome classes.")
    return d.copy()


def design_summary(data: pd.DataFrame, formula: str) -> pd.DataFrame:
    _, x_df = design_from_formula(data, formula)
    rank = int(np.linalg.matrix_rank(x_df.to_numpy(dtype=float)))
    return pd.DataFrame(
        [
            {
                "rows": len(data),
                "parameters": x_df.shape[1],
                "rank": rank,
                "full_rank": rank == x_df.shape[1],
                "candidate": int(data["candidate"].sum()),
                "background": int((data["candidate"] == 0).sum()),
            }
        ]
    )

In [10]:
real_results = []

node_formula = sselib.make_formula("candidate", "sex_entropy_z", PRIMARY_ADJUSTERS)
node_sample = pipeline_like_model_frame(
    node_df,
    predictors=["sex_entropy_z"],
    adjusters=PRIMARY_ADJUSTERS,
    sample_per_class=1200,
)
display(design_summary(node_sample, node_formula))
comparison, summary = show_comparison(
    node_sample,
    node_formula,
    label="sse_node_sex_entropy_primary_sample",
    max_rows=25,
)
real_results.append(summary)

,rows,parameters,rank,full_rank,candidate,background
0,1814,77,77,True,614,1200


,label,formula,nobs,n_parameters,design_rank,full_rank,custom_converged,firthmodels_converged,custom_iterations,firthmodels_iterations,max_abs_estimate_delta,max_abs_std_error_delta,max_abs_p_value_delta
0,sse_node_sex_entropy_primary_sample,candidate ~ sex_entropy_z + C(window_idx) + C(...,1814,77,77,True,True,True,19,12,5.903948e-07,0.880718,0.13662


,custom_estimate,firthmodels_estimate,custom_std_error,firthmodels_std_error,custom_p_value,firthmodels_p_value,estimate_delta,std_error_delta,p_value_delta,abs_estimate_delta,abs_std_error_delta
C(window_idx)[T.6],-1.036575,-1.036575,1.880790,1.540136,0.581539,0.500921,-0.000001,0.340654,0.080618,0.000001,0.340654
C(window_idx)[T.14],-3.568329,-3.568328,2.469927,2.108418,0.148539,0.090566,-0.000001,0.361509,0.057973,0.000001,0.361509
C(window_idx)[T.62],-6.108677,-6.108676,3.019995,2.622302,0.043100,0.019832,-0.000001,0.397693,0.023267,0.000001,0.397693
C(window_idx)[T.65],-7.144882,-7.144881,3.104477,2.743772,0.021365,0.009213,-0.000001,0.360705,0.012151,0.000001,0.360705
C(window_idx)[T.64],-7.112372,-7.112371,3.104560,2.743844,0.021967,0.009539,-0.000001,0.360716,0.012428,0.000001,0.360716
C(window_idx)[T.60],-7.063743,-7.063742,2.847535,2.510282,0.013114,0.004894,-0.000001,0.337253,0.008220,0.000001,0.337253
C(window_idx)[T.16],-4.442258,-4.442257,2.423654,2.091903,0.066821,0.033708,-0.000001,0.331751,0.033113,0.000001,0.331751
C(window_idx)[T.25],-4.716380,-4.716379,2.463740,2.134145,0.055580,0.027108,-0.000001,0.329596,0.028472,0.000001,0.329596
C(window_idx)[T.33],-4.658954,-4.658953,2.464597,2.135665,0.058711,0.029146,-0.000001,0.328932,0.029565,0.000001,0.328932
C(window_idx)[T.38],-5.247805,-5.247805,2.463483,2.134475,0.033152,0.013948,-0.000001,0.329008,0.019204,0.000001,0.329008


In [11]:
sex_term = sselib.categorical_term("sex", reference="Male")
composition_formula = sselib.make_formula("candidate", sex_term, PRIMARY_ADJUSTERS)
composition_sample = pipeline_like_model_frame(
    composition_df,
    predictors=["sex"],
    adjusters=PRIMARY_ADJUSTERS,
    sample_per_class=2000,
    categorical_predictors=True,
)
display(design_summary(composition_sample, composition_formula))
comparison, summary = show_comparison(
    composition_sample,
    composition_formula,
    label="sse_composition_sex_primary_sample",
    max_rows=25,
)
real_results.append(summary)

,rows,parameters,rank,full_rank,candidate,background
0,3985,65,65,True,1998,1987


/opt/homebrew/Caskroom/miniconda/base/envs/PhD/lib/python3.13/site-packages/firthmodels/logistic.py:307: ConvergenceWarning: Step-halving failed to converge.
  result = newton_raphson(


,label,formula,nobs,n_parameters,design_rank,full_rank,custom_converged,firthmodels_converged,custom_iterations,firthmodels_iterations,max_abs_estimate_delta,max_abs_std_error_delta,max_abs_p_value_delta
0,sse_composition_sex_primary_sample,"candidate ~ C(sex, Treatment(reference='Male')...",3985,65,65,True,True,False,20,14,0.000012,0.921086,0.088425


,custom_estimate,firthmodels_estimate,custom_std_error,firthmodels_std_error,custom_p_value,firthmodels_p_value,estimate_delta,std_error_delta,p_value_delta,abs_estimate_delta,abs_std_error_delta
C(clade)[T.21A],3.863029,3.863017,3.213769,2.695398,0.229354,0.151804,0.000012,0.518371,0.077551,0.000012,0.518371
C(clade)[T.22D],7.961866,7.961854,4.469124,3.548038,0.074826,0.024831,0.000011,0.921086,0.049995,0.000011,0.921086
C(clade)[T.22E],7.574992,7.574981,3.701402,3.016646,0.040705,0.012037,0.000011,0.684756,0.028668,0.000011,0.684756
C(clade)[T.20I],4.135137,4.135126,2.527563,2.176413,0.101836,0.057437,0.000011,0.351150,0.044399,0.000011,0.351150
C(clade)[T.22A],6.180037,6.180026,2.807797,2.473375,0.027734,0.012468,0.000011,0.334421,0.015266,0.000011,0.334421
C(clade)[T.21L],7.568086,7.568075,2.579961,2.235766,0.003353,0.000712,0.000011,0.344194,0.002641,0.000011,0.344194
C(clade)[T.recombinant],6.504073,6.504062,2.778513,2.425416,0.019240,0.007327,0.000011,0.353096,0.011914,0.000011,0.353096
C(clade)[T.21K],7.458814,7.458803,2.574850,2.229910,0.003770,0.000823,0.000011,0.344940,0.002947,0.000011,0.344940
C(clade)[T.22B],8.648214,8.648203,2.714644,2.374063,0.001444,0.000270,0.000011,0.340582,0.001174,0.000011,0.340582
C(clade)[T.21J],6.024461,6.024450,2.550455,2.202296,0.018171,0.006228,0.000011,0.348159,0.011943,0.000011,0.348159


## Summary Table

For a correct implementation-level match, the maximum absolute deltas should be close to numerical tolerance. Small differences can still occur because the two solvers use different stopping criteria, step controls, and covariance calculations.

In [12]:
summary_table = pd.concat([*synthetic_results, *real_results], ignore_index=True)
summary_table["estimate_agrees_1e_4"] = summary_table["max_abs_estimate_delta"].le(1e-4)
summary_table["std_error_agrees_1e_4"] = summary_table["max_abs_std_error_delta"].le(1e-4)
summary_table["p_value_agrees_1e_4"] = summary_table["max_abs_p_value_delta"].le(1e-4)
summary_table.sort_values("max_abs_estimate_delta", ascending=False)

,label,formula,nobs,n_parameters,design_rank,full_rank,custom_converged,firthmodels_converged,custom_iterations,firthmodels_iterations,max_abs_estimate_delta,max_abs_std_error_delta,max_abs_p_value_delta,estimate_agrees_1e_4,std_error_agrees_1e_4,p_value_agrees_1e_4
4,sse_composition_sex_primary_sample,"candidate ~ C(sex, Treatment(reference='Male')...",3985,65,65,True,True,False,20,14,1.181436e-05,0.921086,0.088425,True,False,False
1,quasi_separation_with_categorical_term,"y ~ x + C(group, Treatment(reference='A'))",12,5,5,True,True,True,22,9,1.321436e-06,0.734646,0.097443,True,False,False
3,sse_node_sex_entropy_primary_sample,candidate ~ sex_entropy_z + C(window_idx) + C(...,1814,77,77,True,True,True,19,12,5.903948e-07,0.880718,0.136620,True,False,False
0,complete_separation,y ~ x,6,2,2,True,True,True,13,6,2.155916e-07,0.330764,0.066936,True,False,False
2,regular_small_sample,y ~ x1 + x2,80,3,3,True,True,True,6,8,1.177956e-08,0.007419,0.002998,True,False,False


## Interpretation Notes

- This notebook validates agreement conditional on the same Patsy design matrix; it does not validate model specification choices.
- `firthmodels` is treated as the reference implementation here because it is an external maintained package.
- Estimate agreement and standard-error/p-value agreement are intentionally checked separately. Matching estimates with mismatched standard errors usually means the optimisation target agrees but the variance convention differs.
- If large coefficient deltas appear only in rank-deficient designs, inspect the `full_rank` column first. Firth's finite-estimate guarantee assumes a full-rank model matrix.
- If estimates agree but standard errors differ, inspect whether the custom implementation should use the same Firth penalised information convention as `firthmodels` before relying on Wald inference.